# Chapter 31: Greedy Algorithms and Backtracking

The programs from Chapter 31 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## Greedy: Scheduling Meetings

`greedy.py`

In [1]:
"""Greedy algorithms: make the choice that looks best right now, and
never reconsider. Fast and simple, but only right when a proof (or a
counterexample search) says so."""

import heapq


def most_meetings(meetings):
    """The most non-overlapping (start, end) meetings in one room: always
    take the meeting that ends first. Ending early leaves the most
    room for everything after."""
    chosen, free_at = [], float("-inf")
    for start, end in sorted(meetings, key=lambda m: m[1]):
        if start >= free_at:
            chosen.append((start, end))
            free_at = end
    return chosen


def merge_intervals(intervals):
    """Merge overlapping intervals, sorted by start."""
    out = []
    for start, end in sorted(intervals):
        if out and start <= out[-1][1]:
            out[-1][1] = max(out[-1][1], end)
        else:
            out.append([start, end])
    return out


def rooms_needed(meetings):
    """The fewest rooms for all meetings: a heap holds the end times of
    meetings in progress; reuse the room that frees up first."""
    ends = []
    for start, end in sorted(meetings):
        if ends and ends[0] <= start:
            heapq.heapreplace(ends, end)
        else:
            heapq.heappush(ends, end)
    return len(ends)


def can_reach_end(jumps):
    """Each number is the longest jump from that spot. Can we reach the
    last spot? Track the furthest reachable position."""
    furthest = 0
    for i, j in enumerate(jumps):
        if i > furthest:
            return False
        furthest = max(furthest, i + j)
    return True


def greedy_coins(coins, amount):
    """Take the biggest coin that fits, again and again."""
    count = 0
    for c in sorted(coins, reverse=True):
        count += amount // c
        amount %= c
    return count if amount == 0 else -1

## Greedy: How Far Can You Jump?

`greedy_demo.py`

In [2]:
"""Greedy choices on small examples, and one where greedy fails."""

from greedy import (
    can_reach_end,
    greedy_coins,
    merge_intervals,
    most_meetings,
    rooms_needed,
)

meetings = [(9, 10), (9, 12), (10, 11), (11, 13), (12, 14), (13, 15)]
print(most_meetings(meetings))
print(rooms_needed(meetings), "rooms")
print(merge_intervals([(1, 3), (8, 10), (2, 6), (15, 18)]))
print(can_reach_end([2, 3, 1, 1, 4]), can_reach_end([3, 2, 1, 0, 4]))

# US coins: greedy is optimal. Coins 1, 3, 4: it is not.
print(greedy_coins([1, 5, 10, 25], 63), "coins for 63 cents")
print(greedy_coins([1, 3, 4], 6), "coins for 6 with 1, 3, 4 (3 + 3 is 2)")

[(9, 10), (10, 11), (11, 13), (13, 15)]
2 rooms
[[1, 6], [8, 10], [15, 18]]
True False
6 coins for 63 cents
3 coins for 6 with 1, 3, 4 (3 + 3 is 2)


## Backtracking: Trying Everything, Carefully

`backtracking.py`

In [3]:
"""Backtracking: build a solution one choice at a time; when a choice
can't lead anywhere, undo it and try the next."""


def subsets(items):
    out, chosen = [], []

    def go(i):
        if i == len(items):
            out.append(list(chosen))
            return
        go(i + 1)  # leave items[i] out
        chosen.append(items[i])  # put it in
        go(i + 1)
        chosen.pop()  # undo, so the caller sees what it had

    go(0)
    return out


def permutations(items):
    out, chosen, used = [], [], [False] * len(items)

    def go():
        if len(chosen) == len(items):
            out.append(list(chosen))
            return
        for i, x in enumerate(items):
            if not used[i]:
                used[i] = True
                chosen.append(x)
                go()
                chosen.pop()
                used[i] = False

    go()
    return out


def combination_sum(candidates, target):
    """Every combination (repeats allowed) adding up to target."""
    out, chosen = [], []
    candidates = sorted(candidates)

    def go(start, remaining):
        if remaining == 0:
            out.append(list(chosen))
            return
        for i in range(start, len(candidates)):
            c = candidates[i]
            if c > remaining:
                break  # sorted: every later candidate is bigger too
            chosen.append(c)
            go(i, remaining - c)  # i, not i + 1: c may be used again
            chosen.pop()

    go(0, target)
    return out


def queens(n):
    """Ways to place n queens on an n x n board, none attacking another.
    One queen per row; sets of used columns and diagonals prune fast."""
    cols, diag, anti = set(), set(), set()

    def place(row):
        if row == n:
            return 1
        count = 0
        for c in range(n):
            if c in cols or row - c in diag or row + c in anti:
                continue  # attacked: prune this whole branch
            cols.add(c)
            diag.add(row - c)
            anti.add(row + c)
            count += place(row + 1)
            cols.remove(c)
            diag.remove(row - c)
            anti.remove(row + c)
        return count

    return place(0)

## The N-Queens Puzzle

`backtracking_demo.py`

In [4]:
"""Backtracking on small examples."""

from backtracking import combination_sum, permutations, queens, subsets

print(["".join(s) for s in subsets("abc")])
print(permutations([1, 2, 3]))
print(combination_sum([2, 3, 5], 8))
print([queens(n) for n in range(1, 11)])

['', 'c', 'b', 'bc', 'a', 'ac', 'ab', 'abc']
[[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]
[[2, 2, 2, 2], [2, 3, 3], [3, 5]]
[1, 0, 0, 2, 10, 4, 40, 92, 352, 724]


## Timing

`bench_queens.py`

In [5]:
"""Counting n-queens placements: backtracking with pruning still grows
exponentially. Each row of the table adds one to n."""

import bench
from backtracking import queens

sizes = [6, 7, 8, 9, 10]
bench.table({"queens(n)": bench.doubling(queens, lambda n: (n,), sizes)})

         n       queens(n)
         6         67.8 us
         7     393 us x5.8
         8    1.69 ms x4.3
         9    7.06 ms x4.2
        10    31.7 ms x4.5
